# Export Backend Comparison: ONNX vs OpenVINO vs ExecuTorch

This notebook demonstrates the full pipeline:
1. Train an ACT policy on the PushT dataset
2. Export to multiple backends (ONNX, OpenVINO, ExecuTorch)
3. Load exported models for inference
4. Compare numerical consistency across backends
5. Benchmark inference latency (mean, p50, p99)

In [ ]:
# Copyright (C) 2025 Intel Corporation
# SPDX-License-Identifier: Apache-2.0

import time
import logging
from pathlib import Path

import numpy as np
import torch

from physicalai.data import LeRobotDataModule
from physicalai.data.lerobot import FormatConverter
from physicalai.inference import InferenceModel
from physicalai.policies import ACT
from physicalai.train import Trainer

logging.basicConfig(level=logging.INFO)

EXPORT_DIR = Path("./exports")
EXPORT_DIR.mkdir(exist_ok=True)

## 1. Train ACT Policy

We train an ACT policy on the PushT dataset using `fast_dev_run=1` for a quick demonstration.
In production, increase `max_epochs` and remove `fast_dev_run`.

In [ ]:
datamodule = LeRobotDataModule(
    repo_id="lerobot/pusht",
    train_batch_size=8,
    episodes=list(range(10)),
)
policy = ACT()
trainer = Trainer(fast_dev_run=1, enable_checkpointing=False, logger=False, enable_progress_bar=False)
trainer.fit(model=policy, datamodule=datamodule)
policy.eval()
print("Training complete!")

## 2. Export to All Backends

Export the trained policy to ONNX, OpenVINO, and ExecuTorch.
Each backend is wrapped in `try/except` so the notebook continues even if
an optional dependency (e.g., `executorch`) is not installed.

In [ ]:
backends = ["onnx", "openvino", "executorch"]
results = {}

for backend in backends:
    export_path = EXPORT_DIR / backend
    try:
        start = time.perf_counter()
        policy.export(export_path, backend)
        export_time = time.perf_counter() - start
        results[backend] = {"status": "success", "export_time": export_time, "path": export_path}
        print(f"\u2713 {backend}: exported in {export_time:.2f}s")
    except ImportError as e:
        results[backend] = {"status": "skipped", "reason": str(e)}
        print(f"\u2298 {backend}: skipped (optional dependency not installed: {e})")
    except Exception as e:
        results[backend] = {"status": "failed", "reason": str(e)}
        print(f"\u2717 {backend}: failed ({e})")

## 3. Load Inference Models

Prepare inference input using `FormatConverter.to_observation()` (the same pattern
used in integration tests), then load each successfully exported model.

In [ ]:
# Prepare inference input using the same pattern as integration tests
sample_batch = next(iter(datamodule.train_dataloader()))
batch_observation = FormatConverter.to_observation(sample_batch)
inference_input = batch_observation[0:1].to_numpy().to_dict(flatten=False)

# Load each successfully exported model
inference_models = {}
for backend, result in results.items():
    if result["status"] != "success":
        print(f"\u2298 {backend}: skipping load (export {result['status']})")
        continue
    try:
        inference_models[backend] = InferenceModel.load(result["path"])
        print(f"\u2713 {backend}: loaded")
    except ImportError as e:
        print(f"\u2717 {backend}: load failed (missing dependency: {e})")
    except Exception as e:
        print(f"\u2717 {backend}: load failed ({e})")

## 4. Numerical Consistency Check

Compare the output of each exported backend against the original training model.
We use the same tolerance (`rtol=0.2, atol=0.2`) as the integration tests.

In [ ]:
# Get reference output from the trained policy (same pattern as integration tests)
device = next(policy.parameters()).device
torch.manual_seed(42)
single_obs = batch_observation[0:1].to(device)
with torch.no_grad():
    train_action = policy.predict_action_chunk(single_obs)
if isinstance(train_action, tuple):
    train_action = train_action[0]
train_action = train_action.squeeze(0)
if len(train_action.shape) > 1:
    train_action = train_action[0]
print(f"Training output shape: {train_action.shape}")
print(f"Training output (first 5 values): {train_action[:5]}")

In [ ]:
print("Numerical consistency (tolerance: rtol=0.2, atol=0.2)\n")
for backend, model in inference_models.items():
    try:
        torch.manual_seed(42)
        inference_output = model.select_action(inference_input)
        inference_tensor = torch.as_tensor(inference_output).cpu().squeeze(0)
        if len(inference_tensor.shape) > 1:
            inference_tensor = inference_tensor[0]

        max_diff = torch.max(torch.abs(inference_tensor - train_action)).item()
        torch.testing.assert_close(inference_tensor, train_action, rtol=0.2, atol=0.2)
        results[backend]["numerical_pass"] = True
        results[backend]["max_abs_diff"] = max_diff
        print(f"\u2713 {backend}: PASS (max abs diff: {max_diff:.6f})")
    except AssertionError as e:
        results[backend]["numerical_pass"] = False
        print(f"\u2717 {backend}: FAIL ({e})")
    except Exception as e:
        results[backend]["numerical_pass"] = None
        print(f"\u2298 {backend}: error ({e})")

## 5. Latency Benchmark

Measure inference latency for each backend. We run a warmup phase first
to ensure caches are populated and JIT compilation is complete,
then time the benchmark iterations.

In [ ]:
WARMUP_ITERS = 10
BENCH_ITERS = 100

print(f"Benchmarking ({WARMUP_ITERS} warmup + {BENCH_ITERS} timed iterations)\n")
for backend, model in inference_models.items():
    # Warmup
    for _ in range(WARMUP_ITERS):
        model.select_action(inference_input)

    # Benchmark
    latencies_ms = []
    for _ in range(BENCH_ITERS):
        start = time.perf_counter()
        model.select_action(inference_input)
        elapsed_ms = (time.perf_counter() - start) * 1000
        latencies_ms.append(elapsed_ms)

    mean_ms = np.mean(latencies_ms)
    p50_ms = np.percentile(latencies_ms, 50)
    p99_ms = np.percentile(latencies_ms, 99)

    results[backend]["latency_mean"] = mean_ms
    results[backend]["latency_p50"] = p50_ms
    results[backend]["latency_p99"] = p99_ms
    print(f"{backend:12s}: mean={mean_ms:.2f}ms  p50={p50_ms:.2f}ms  p99={p99_ms:.2f}ms")

## 6. Summary

A consolidated view of export status, latency, and numerical consistency across all backends.

In [ ]:
# Print summary as a markdown-style pipe table
header = "| Backend     | Status  | Export Time | Mean Latency | P50    | P99    | Max Abs Diff |"
separator = "|-------------|---------|-------------|--------------|--------|--------|--------------|"
print(header)
print(separator)
for backend, r in results.items():
    status = r.get("status", "?")
    export_t = f"{r['export_time']:.2f}s" if "export_time" in r else "-"
    mean = f"{r['latency_mean']:.2f}ms" if "latency_mean" in r else "-"
    p50 = f"{r['latency_p50']:.2f}ms" if "latency_p50" in r else "-"
    p99 = f"{r['latency_p99']:.2f}ms" if "latency_p99" in r else "-"
    max_diff = f"{r['max_abs_diff']:.4f}" if "max_abs_diff" in r else "-"
    print(f"| {backend:<11} | {status:<7} | {export_t:<11} | {mean:<12} | {p50:<6} | {p99:<6} | {max_diff:<12} |")
